# Update selected sources
Choose the state/product pairs you want to refresh. Each collector uses its existing history routine;
this notebook selects sources, not individual reporting months. No incremental-download system is involved.

The download/write cell is disabled by default, so **Run All** first shows the plan and stored coverage.
For everyday analysis, open `90_consolidated_ggr.ipynb`.

In [ ]:
database_file = "data/staging/gaming_nationwide.sqlite"  # selected write destination
selected_sources = [("MA", "online_sports_betting")]
# Set selected_sources = None to run the complete registered inventory.
run_downloads = False  # Change to True only when you want to download and write data.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.collect import COLLECTORS, inventory_collector_order, run_all_collectors
from variant_gaming.storage import connect_readonly, default_db_path

database_path = ROOT / database_file
print(f"Download/write destination: {database_path}")

## 1. Check the plan and destination
Pairs are `(state_code, product)`: `online_sports_betting` or `online_casino`.
The inventory includes all 102 combinations; only implemented collectors and specific gap recorders run.
Gap recorders copy dated research notes and produce no revenue rows. They do not perform a fresh source check.

This task uses `data/staging/gaming_nationwide.sqlite`. Analysis notebook 90 can select that same file.
Writing to `data/gaming.sqlite` is a separate, later reviewed action; see README.

In [ ]:
scheduled = inventory_collector_order(ROOT)
if selected_sources is not None:
    requested = {(state.upper(), product) for state, product in selected_sources}
    if not requested or requested - COLLECTORS.keys():
        raise ValueError("Choose nonempty, registered state/product pairs")
    scheduled = [row for row in scheduled if (row[1], row[2]) in requested]
plan = pd.DataFrame(scheduled, columns=["wave", "state_code", "vertical"])
display(plan)

## 2. Download and write
Set `run_downloads = True` to download official reports, retain new captures and write to the displayed database.
Current collectors run history routines. Review both run status and coverage status afterward.
For new PDF collectors, per-report successes and errors are also saved beside the selected database.

In [ ]:
summary = pd.DataFrame()
if run_downloads:
    summary = run_all_collectors(root=ROOT, db_path=database_path, selected=selected_sources)
    display(summary)
else:
    print("Downloads are disabled. Set run_downloads=True to execute the plan above.")

In [ ]:
if not summary.empty:
    needs_attention = summary[
        summary["run_status"].ne("completed") | summary["coverage_status"].ne("ok")
    ]
    display(needs_attention[["state_code", "vertical", "run_status", "coverage_status",
                             "returned_rows", "database_rows", "coverage_reason", "run_error"]])

## 3. Inspect the stored coverage
This cell reads SQLite without changing it. A failed refresh can leave older observations available.
The analysis notebook reads the database directly; a CSV rebuild is not required.

In [ ]:
if database_path.exists():
    connection = connect_readonly(database_path)
    try:
        coverage = pd.read_sql_query("SELECT * FROM source_coverage", connection)
    finally:
        connection.close()
    display(plan.merge(coverage, on=["state_code", "vertical"], how="left"))
else:
    print("No local database yet. A successful collection creates it.")